# Homework 2: Multiple Object Video Detection
## Arsen Botsko, Daryna Antoniuk

In [ ]:
%pip install -q ultralytics==8.4.190 pycocotools wandb

In [ ]:
import os, sys, gc, json, time, shutil, random, subprocess, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from contextlib import redirect_stdout
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torchvision, wandb, yaml, ultralytics
import torch.nn.functional as F
from torch import nn
from PIL import Image, ImageDraw
from sklearn.model_selection import GroupKFold
from sklearn.cluster import KMeans
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision.io import decode_image, ImageReadMode
from torchvision.ops import box_iou, box_convert, batched_nms, generalized_box_iou_loss
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, retinanet_resnet50_fpn_v2, roi_heads, retinanet as tv_retinanet, _utils as det_utils
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.image_list import ImageList
from torchvision.models.detection.transform import resize_boxes
from ultralytics import YOLO, RTDETR

START = time.time()
COLAB = 'google.colab' in sys.modules
KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/AI_HW2')

def kaggle_input(name):
    return [p for d in ['*', '*/*', '*/*/*', '*/*/*/*', '*/*/*/*/*'] for p in Path('/kaggle/input').glob(f'{d}/{name}')]

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DATA = kaggle_input('test_images.csv')[0].parent if KAGGLE else Path('data')
DS = Path('/content/ds') if COLAB else Path('/tmp/ds') if KAGGLE else Path('work/ds')
PROJECT = 'ai-hw2'
RUNS = Path(PROJECT)
PREV_KERNEL = 'starezax20/notebookd0ef990b98/6'
if KAGGLE and PREV_KERNEL:
    from kaggle_secrets import UserSecretsClient
    os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle==2.2.5'], check=True)
    subprocess.run([sys.executable, '-m', 'kaggle', 'kernels', 'output', PREV_KERNEL, '-p', '/tmp/prev', '--file-pattern', f'^{PROJECT}/', '-q'], check=True)
    shutil.copytree(Path('/tmp/prev') / PROJECT, RUNS, dirs_exist_ok=True)
    assert any(RUNS.glob('*/val_metrics.json')), f'nothing restored from {PREV_KERNEL}'
print('runs from earlier sessions:', sorted(p.parent.name for p in RUNS.glob('*/val_metrics.json')) or 'none')
RUNS.mkdir(exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPUS = torch.cuda.device_count()
TIME_LIMIT_H = 7 if KAGGLE else None

CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
IMGSZ, EPOCHS = 1024, 8
TRAIN_STEP, VAL_STEP = 10, 10
CONF, NMS_IOU, MAX_DET = 0.001, 0.6, 500
print(os.getcwd(), '|', f'{GPUS} x {torch.cuda.get_device_name()}' if GPUS else 'cpu', '| torch', torch.__version__, '| ultralytics', ultralytics.__version__)

In [ ]:
def wandb_key():
    try:
        if COLAB:
            from google.colab import userdata
            return userdata.get('WANDB_API_KEY')
        if KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret('WANDB_API_KEY')
    except Exception:
        pass
    return os.environ.get('WANDB_API_KEY', '')

os.environ['WANDB_API_KEY'] = wandb_key() or ''
WANDB_MODE = 'online' if os.environ['WANDB_API_KEY'] else 'disabled'
os.environ['WANDB_MODE'] = WANDB_MODE
ultralytics.settings.update({'wandb': WANDB_MODE != 'disabled'})
print('W&B logging:', WANDB_MODE)

## 1. Data Understanding

### Annotations

In [ ]:
VISDRONE = ['ignored region', 'pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor', 'others']

def load_split(split):
    d = json.load(open(DATA / split / 'annotations_coco.json'))
    imgs = pd.DataFrame(d['images']).rename(columns={'id': 'coco_id', 'frame_id': 'frame'})
    imgs['file'] = split + '/' + imgs.file_name
    imgs['path'] = [str(DATA / f) for f in imgs.file]
    ann = pd.DataFrame(d['annotations']).merge(imgs[['coco_id', 'video', 'frame']], left_on='image_id', right_on='coco_id')
    return imgs[['video', 'frame', 'width', 'height', 'file', 'path']], ann.drop(columns=['id', 'image_id', 'coco_id'])

(f1, a1), (f2, a2) = load_split('train'), load_split('val')
frames = pd.concat([f1, f2], ignore_index=True)
frames['image_id'] = frames.index
frames['key'] = frames.video + '_' + frames.frame.map('{:07d}'.format)
ann = pd.concat([a1, a2], ignore_index=True).merge(frames[['video', 'frame', 'image_id', 'width', 'height']])
ann[['x', 'y', 'w', 'h']] = np.array(ann.bbox.tolist(), dtype=float)
ann = ann.drop(columns=['bbox', 'ignore'], errors='ignore')
objs =ann[ann.iscrowd == 0].assign(cls=lambda d: d.category_id - 1, side=lambda d: np.sqrt(d.w * d.h))
ignore = ann[(ann.iscrowd == 1) & (ann.category_id == 1)]
del f1, f2, a1, a2

print(f'{frames.video.nunique()} clips | {len(frames)} frames | {len(objs)} objects | {len(ignore)} ignored regions')
for k, c in enumerate(CLASSES):
    print(f'{c:9s} <- ' + ' + '.join(VISDRONE[i] for i in sorted(objs.orig_category_id[objs.cls == k].unique())))
print('ignored  <- ' + ' + '.join(VISDRONE[i] for i in sorted(ignore.orig_category_id.unique())))

### Clips and resolutions

In [ ]:
clips = frames.groupby('video').agg(frames=('frame', 'size'), width=('width', 'first'))
test = pd.read_csv(DATA / 'test_images.csv')
print(f'clip length: {clips.frames.min()}-{clips.frames.max()} frames, median {clips.frames.median():.0f} | test clips: {test["clip"].nunique()}, median {test.groupby("clip").size().median():.0f} frames')
pd.concat({'train+val frames': frames.groupby(['width', 'height']).size(), 'test frames': test.groupby(['width', 'height']).size()}, axis=1).fillna(0).astype(int)

### Class balance

In [ ]:
shares = pd.crosstab(objs.video, objs.cls, normalize='index').set_axis(CLASSES, axis=1)
counts = objs.cls.value_counts().sort_index().set_axis(CLASSES)
fig, ax = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={'width_ratios': [1, 2.5]})
ax[0].bar(CLASSES, counts, color=COLORS)
ax[0].set(yscale='log', title='objects per class (log scale)')
ax[0].tick_params(axis='x', rotation=45)
im = ax[1].imshow(shares.sort_values('car').T, aspect='auto', cmap='Blues')
ax[1].set(yticks=range(8), yticklabels=CLASSES, xticks=[], xlabel='clip (sorted by the share of cars)', title='class share inside each clip')
fig.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()
print('share of all objects, %:', (counts / counts.sum() * 100).round(1).to_dict())
print('clips without a bus:', (shares.bus == 0).sum(), '| without a truck:', (shares.truck == 0).sum(), '| without a tricycle:', (shares.tricycle == 0).sum())

### Object sizes

In [ ]:
objs['side_in'] = objs.side * IMGSZ / objs[['width', 'height']].max(axis=1)
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].hist(objs.side, bins=np.logspace(0.5, 3.2, 60), color=COLORS[0])
ax[0].set(xscale='log', title='sqrt(box area), original pixels', xlabel='px')
ax[1].hist(objs.side_in, bins=np.logspace(0, 2.7, 60), color=COLORS[0])
for v in [8, 16, 32]:
    ax[1].axvline(v, color='gray', ls='--', lw=1)
ax[1].set(xscale='log', title=f'sqrt(box area) at long side {IMGSZ}', xlabel='px')
ax[2].boxplot([objs.side_in[objs.cls == k] for k in range(8)], showfliers=False)
ax[2].set_xticks(range(1, 9), CLASSES, rotation=45)
ax[2].set(yscale='log', title=f'size per class at long side {IMGSZ}, px')
plt.tight_layout(); plt.show()

area = objs.w * objs.h
print('COCO size groups (original pixels): small %.1f%% | medium %.1f%% | large %.1f%%' % tuple(100 * np.array([(area < 32**2).mean(), ((area >= 32**2) & (area < 96**2)).mean(), (area >= 96**2).mean()])))
print('median height / width:', (objs.h / objs.w).groupby(objs.cls).median().set_axis(CLASSES).round(2).to_dict())
long_side = objs[['width', 'height']].max(axis=1)
pd.DataFrame({f'long side {s}': objs.side * s / long_side for s in [640, 1024, 1280, 1920]}).quantile([.05, .25, .5, .75, .95]).round(1).rename_axis('quantile of sqrt(area), px')

### Objects per frame

In [ ]:
per_frame = objs.groupby('image_id').size().reindex(frames.image_id, fill_value=0)
by_clip = per_frame.groupby(frames.video.values)
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={'width_ratios': [1, 2.5]})
ax[0].hist(per_frame, bins=60, color=COLORS[0])
ax[0].set(title='objects per frame', xlabel='objects')
ax[1].boxplot([by_clip.get_group(c) for c in by_clip.median().sort_values().index], showfliers=False)
ax[1].set(xticks=[], xlabel='clip (sorted by median)', title='objects per frame inside each clip')
plt.tight_layout(); plt.show()
print('objects per frame: median %d | 90%% %d | 99%% %d | max %d' % tuple(per_frame.quantile([.5, .9, .99, 1])))

### Occlusion and truncation

In [ ]:
print(f'truncated objects: {objs.truncation.mean():.1%}')
(pd.crosstab(objs.cls, objs.occlusion, normalize='index').set_axis(CLASSES).set_axis(['none', 'partial', 'heavy'], axis=1) * 100).round(1)

### Ignored regions

In [ ]:
def inter(m, s):
    ix = np.minimum(m.x + m.w, m['x' + s] + m['w' + s]) - np.maximum(m.x, m['x' + s])
    iy = np.minimum(m.y + m.h, m['y' + s] + m['h' + s]) - np.maximum(m.y, m['y' + s])
    return ix.clip(lower=0) * iy.clip(lower=0)

def covered(boxes, regions):
    m = boxes[['image_id', 'x', 'y', 'w', 'h']].reset_index(names='row').merge(regions[['image_id', 'x', 'y', 'w', 'h']], on='image_id', suffixes=('', '_r'))
    return (inter(m, '_r') / (m.w * m.h)).groupby(m.row).sum().clip(upper=1).reindex(boxes.index, fill_value=0).to_numpy()

objs['ignored'] = covered(objs, ignore)
ignored_area = (ignore.w * ignore.h).groupby(ignore.image_id).sum() / frames.set_index('image_id').eval('width * height')
print(f'frames with ignored regions: {ignore.image_id.nunique() / len(frames):.0%} | share of the frame they cover: median {ignored_area.median():.1%}, 90% {ignored_area.quantile(.9):.1%}')
print(f'objects partly inside ignored regions: {(objs.ignored > 0).mean():.1%} | more than half inside: {(objs.ignored > .5).mean():.2%}')

### How fast the frames change

In [ ]:
def lag_iou(k):
    a = objs[['video', 'track_id', 'frame', 'x', 'y', 'w', 'h']]
    m = a.merge(a.assign(frame=a.frame - k), on=['video', 'track_id', 'frame'], suffixes=('', '_k'))
    i = inter(m, '_k')
    return i / (m.w * m.h + m.w_k * m.h_k - i)

lags = [1, 2, 3, 5, 10, 15, 20, 30]
ious = {k: lag_iou(k) for k in lags}
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(lags, [ious[k].median() for k in lags], 'o-', color=COLORS[0], label='median IoU')
ax.plot(lags, [(ious[k] > .5).mean() for k in lags], 's-', color=COLORS[1], label='share with IoU > 0.5')
ax.set(xlabel='frames apart', title='same object k frames later')
ax.legend(); plt.show()
print({k: round(ious[k].median(), 2) for k in lags})

### Ground truth on sample frames

In [ ]:
def draw(ax, path, boxes=(), colors=(), regions=(), crop=None, title=''):
    ax.imshow(Image.open(path))
    for (x, y, w, h), c in zip(boxes, colors):
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, ec=c, lw=0.8))
    for x, y, w, h in regions:
        ax.add_patch(plt.Rectangle((x, y), w, h, fc='white', ec='white', alpha=0.35, hatch='///'))
    if crop is not None:
        ax.set(xlim=(crop[0], crop[2]), ylim=(crop[3], crop[1]))
    ax.set_title(title, fontsize=9)
    ax.axis('off')

def gt_of(image_id):
    o, r = objs[objs.image_id == image_id], ignore[ignore.image_id == image_id]
    return o[['x', 'y', 'w', 'h']].values, [COLORS[c] for c in o.cls], r[['x', 'y', 'w', 'h']].values

sample = frames.groupby('video').sample(1, random_state=SEED).sample(5, random_state=SEED)
dense = sample.loc[per_frame[sample.image_id].idxmax()]
o = objs[objs.image_id == dense.image_id]
cx, cy = (o.x + o.w / 2).median(), (o.y + o.h / 2).median()
fig, ax = plt.subplots(2, 3, figsize=(18, 7))
for a, f in zip(ax.flat, sample.itertuples()):
    draw(a, f.path, *gt_of(f.image_id), title=f'{f.key} ({f.width}x{f.height}): {per_frame[f.image_id]} objects')
draw(ax.flat[-1], dense.path, *gt_of(dense.image_id), crop=(cx - dense.width / 8, cy - dense.height / 8, cx + dense.width / 8, cy + dense.height / 8), title=f'zoom into {dense.key}')
fig.legend(handles=[plt.Line2D([], [], color=c, label=n) for c, n in zip(COLORS, CLASSES)], loc='lower center', ncol=8)
plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()

### What the EDA tells

## 2. Preprocessing

### Split by clip

In [ ]:
def class_shares(o):
    return o.cls.value_counts(normalize=True).reindex(range(8), fill_value=0)

folds = [h for _, h in GroupKFold(5).split(frames, groups=frames.video)]
mismatch = [np.abs(np.log(class_shares(objs[objs.image_id.isin(h)]) / class_shares(objs[~objs.image_id.isin(h)]))).sum() for h in folds]
holdout_clips = frames.video[folds[int(np.argmin(mismatch))]].unique()
frames['holdout'] = frames.video.isin(holdout_clips)
objs['holdout'] = objs.video.isin(holdout_clips)
print('class mismatch of the 5 folds:', np.round(mismatch, 2), '-> fold', np.argmin(mismatch))

pd.DataFrame({part: {'clips': fr.video.nunique(), 'frames': len(fr), 'objects': int((objs.holdout == h).sum()), **(class_shares(objs[objs.holdout == h]) * 100).round(1).set_axis(CLASSES)}
              for part, h, fr in [('fit', False, frames[~frames.holdout]), ('hold-out', True, frames[frames.holdout])]})

### Frame subsampling and conversion

In [ ]:
frames['val'] = frames.holdout & ((frames.frame - 1) % VAL_STEP == 0)
VAL = frames[frames.val]

def convert(f, boxes, regions):
    out = DS / 'images' / f'{f.key}.jpg'
    if out.exists():
        return
    img = Image.open(f.path).convert('RGB')
    paint = ImageDraw.Draw(img)
    for x, y, w, h in regions:
        paint.rectangle([x, y, x + w, y + h], fill=(114, 114, 114))
    lines = [f'{int(c)} {(x + w / 2) / f.width:.6f} {(y + h / 2) / f.height:.6f} {w / f.width:.6f} {h / f.height:.6f}' for c, x, y, w, h in boxes]
    (DS / 'labels' / f'{f.key}.txt').write_text('\n'.join(lines))
    img.save(out, quality=95)

def prepare(step):
    (DS / 'images').mkdir(parents=True, exist_ok=True)
    (DS / 'labels').mkdir(exist_ok=True)
    sel = frames[(frames.frame - 1) % step == 0]
    keep = objs[objs.image_id.isin(sel.image_id) & (objs.ignored <= .5)]
    boxes = {i: g[['cls', 'x', 'y', 'w', 'h']].to_numpy() for i, g in keep.groupby('image_id')}
    regions = {i: g[['x', 'y', 'w', 'h']].to_numpy() for i, g in ignore[ignore.image_id.isin(sel.image_id)].groupby('image_id')}
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(lambda f: convert(f, boxes.get(f.image_id, []), regions.get(f.image_id, [])), sel.itertuples()))
    for name, fr in {f'fit{step}': sel[~sel.holdout], f'all{step}': sel, 'holdout': VAL}.items():
        (DS / f'{name}.txt').write_text('\n'.join(str((DS / 'images' / f'{k}.jpg').resolve()) for k in fr.key))
        if name != 'holdout':
            yaml.safe_dump({'path': str(DS.resolve()), 'train': f'{name}.txt', 'val': 'holdout.txt', 'names': dict(enumerate(CLASSES))}, open(DS / f'{name}.yaml', 'w'))
    print(f'step {step}: {len(sel)} frames ready | fit {(~sel.holdout).sum()} | validation {len(VAL)}')

prepare(TRAIN_STEP)

### Check the conversion

In [ ]:
def read_labels(key):
    return np.array((DS / 'labels' / f'{key}.txt').read_text().split(), dtype=float).reshape(-1, 5)

check = frames[((frames.frame - 1) % TRAIN_STEP == 0)].assign(ign=lambda d: ignored_area.reindex(d.image_id).fillna(0).values)
check = check.sort_values('ign').groupby('video').tail(1).tail(2)
fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
for a, f in zip(ax, check.itertuples()):
    path = DS / 'images' / f'{f.key}.jpg'
    W, H = Image.open(path).size
    lab = read_labels(f.key)
    boxes = np.c_[(lab[:, 1] - lab[:, 3] / 2) * W, (lab[:, 2] - lab[:, 4] / 2) * H, lab[:, 3] * W, lab[:, 4] * H]
    draw(a, path, boxes, [COLORS[int(c)] for c in lab[:, 0]], title=f'{f.key} after conversion ({W}x{H}, {len(lab)} labels)')
plt.tight_layout(); plt.show()

### Evaluation

In [ ]:
METRICS = ['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']

def coco_gt(fr):
    a = ann[ann.image_id.isin(fr.image_id)]
    return {'images': [{'id': int(i), 'width': int(w), 'height': int(h)} for i, w, h in zip(fr.image_id, fr.width, fr.height)],
            'annotations': [{'id': n + 1, 'image_id': int(i), 'category_id': int(c), 'bbox': [x, y, w, h], 'area': w * h, 'iscrowd': int(z), 'track_id': int(t)}
                            for n, (i, c, x, y, w, h, z, t) in enumerate(a[['image_id', 'category_id', 'x', 'y', 'w', 'h', 'iscrowd', 'track_id']].fillna(-1).itertuples(index=False))],
            'categories': [{'id': k + 1, 'name': c} for k, c in enumerate(CLASSES)]}

frames.to_csv(RUNS / 'frames.csv', index=False)
json.dump(coco_gt(frames[frames.holdout]), open(RUNS / 'holdout_gt.json', 'w'))
with redirect_stdout(None):
    GT = COCO(RUNS / 'holdout_gt.json')

def evaluate(dets, ids=None):
    ids = VAL.image_id.tolist() if ids is None else list(ids)
    if len(dets) == 0:
        return dict.fromkeys(METRICS + CLASSES, 0.0)
    with redirect_stdout(None):
        E = COCOeval(GT, GT.loadRes([dict(d) for d in dets]), 'bbox')
        E.params.imgIds, E.params.maxDets = ids, [1, 100, MAX_DET]
        E.evaluate(); E.accumulate(); E.summarize()
    p = E.eval['precision'][:, :, :, 0, -1]
    return {k: round(float(v), 4) for k, v in zip(METRICS, E.stats)} | {c: round(float(p[..., k][p[..., k] > -1].mean()), 4) for k, c in enumerate(CLASSES)}

print(f'validation: {len(VAL)} frames from {VAL.video.nunique()} clips, {objs.image_id.isin(VAL.image_id).sum()} objects')

### Prediction and logging

In [ ]:
def xyxy(b):
    b = np.asarray(b, dtype=float).reshape(-1, 4)
    return np.c_[b[:, :2], b[:, :2] + b[:, 2:]]

def to_coco(image_id, boxes, scores, labels):
    return [{'image_id': int(image_id), 'category_id': int(c) + 1, 'bbox': [round(x1, 1), round(y1, 1), round(x2 - x1, 1), round(y2 - y1, 1)], 'score': round(s, 4)}
            for (x1, y1, x2, y2), s, c in zip(boxes.tolist(), scores.tolist(), labels.tolist())]

def sync():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

@torch.no_grad()
def predict_tv(model, fr):
    model.eval().to(DEVICE)
    dets, ms = [], []
    for i, path in zip(fr.image_id, fr.path):
        img = decode_image(path, ImageReadMode.RGB).to(DEVICE).float() / 255
        sync(); t = time.perf_counter()
        out = model([img])[0]
        sync(); ms.append((time.perf_counter() - t) * 1000)
        dets += to_coco(i, out['boxes'].cpu().numpy(), out['scores'].cpu().numpy(), out['labels'].cpu().numpy() - 1)
    return dets, float(np.median(ms))

def predict_ultra(model, fr, **kw):
    dets, ms = [], []
    for i, path in zip(fr.image_id, fr.path):
        r = model.predict(path, conf=CONF, iou=NMS_IOU, max_det=MAX_DET, verbose=False, **kw)[0]
        dets += to_coco(i, r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy())
        ms.append(sum(r.speed.values()))
    return dets, float(np.median(ms))

def wandb_image(f, dets, thr=0.3):
    box = lambda b, c, **e: {'position': {'minX': b[0], 'minY': b[1], 'maxX': b[0] + b[2], 'maxY': b[1] + b[3]}, 'domain': 'pixel', 'class_id': int(c), **e}
    g = objs[objs.image_id == f.image_id]
    pred = [box(d['bbox'], d['category_id'] - 1, scores={'score': d['score']}) for d in dets if d['image_id'] == f.image_id and d['score'] >= thr]
    return wandb.Image(f.path, boxes={'predictions': {'box_data': pred, 'class_labels': dict(enumerate(CLASSES))},
                                      'ground_truth': {'box_data': [box(b, c) for b, c in zip(g[['x', 'y', 'w', 'h']].values.tolist(), g.cls)], 'class_labels': dict(enumerate(CLASSES))}})

results = {}

def run_eval(name, model, **kw):
    out = RUNS / name
    if not (out / 'val_metrics.json').exists():
        dets, ms = predict_ultra(model, VAL, **kw) if isinstance(model, (YOLO, RTDETR)) else predict_tv(model, VAL)
        metrics = evaluate(dets) | {'ms/img': round(ms, 1)}
        out.mkdir(parents=True, exist_ok=True)
        json.dump(dets, open(out / 'val_preds.json', 'w'))
        json.dump(metrics, open(out / 'val_metrics.json', 'w'))
        if WANDB_MODE != 'disabled':
            with wandb.init(project=PROJECT, name=name, group=name, job_type='eval') as run:
                run.summary.update(metrics)
                run.log({'val_predictions': [wandb_image(f, dets) for f in VAL.sample(4, random_state=SEED).itertuples()]})
    results[name] = json.load(open(out / 'val_metrics.json'))
    return results[name]

def load_dets(name):
    return json.load(open(RUNS / name / 'val_preds.json'))

def table(names):
    return pd.DataFrame({n: results[n] for n in names}).T[METRICS + ['ms/img']]

def check_time():
    if TIME_LIMIT_H and time.time() - START > TIME_LIMIT_H * 3600:
        print(f'{TIME_LIMIT_H} h of this session are used. Run the notebook again with the output of this version attached as input.')
        raise SystemExit(0)

def tune(steps, fit, cfg=None, best=None):
    cfg = dict(cfg or {})
    for name, change in steps.items():
        run_eval(name, fit(name, **(cfg | change)))
        if best is None or results[name]['mAP'] > results[best]['mAP']:
            cfg, best = cfg | change, name
    print('best run:', best, '| kept settings:', cfg)
    return best, cfg

### The shared protocol

## 3. Two-stage anchor-based: Faster R-CNN

### Training loop for the torchvision models

In [ ]:
class Frames(torch.utils.data.Dataset):
    def __init__(self, list_name, flip=True):
        self.paths, self.flip = (DS / f'{list_name}.txt').read_text().split(), flip

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = decode_image(self.paths[i], ImageReadMode.RGB)
        h, w = img.shape[1:]
        c, cx, cy, bw, bh = read_labels(Path(self.paths[i]).stem).T
        boxes = np.stack([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h], 1)
        if self.flip and torch.rand(1) < 0.5:
            img, boxes = img.flip(-1), np.stack([w - boxes[:, 2], boxes[:, 1], w - boxes[:, 0], boxes[:, 3]], 1)
        return img, {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4), 'labels': torch.tensor(c, dtype=torch.int64) + 1}

def collate(batch):
    return tuple(zip(*batch))

FASTRCNN_LOSS = roi_heads.fastrcnn_loss

def train_tv(name, model, lr, data=f'fit{TRAIN_STEP}', epochs=EPOCHS, batch=4):
    path = RUNS / name / 'model.pt'
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        return model.to(DEVICE).eval()
    check_time()
    path.parent.mkdir(parents=True, exist_ok=True)
    json.dump(model.cfg | {'imgsz': IMGSZ}, open(path.parent / 'config.json', 'w'))
    roi_heads.fastrcnn_loss = giou_fastrcnn_loss if model.cfg.get('giou') else FASTRCNN_LOSS
    loader = torch.utils.data.DataLoader(Frames(data), batch, shuffle=True, num_workers=4, collate_fn=collate, drop_last=True)
    params = [p for p in model.to(DEVICE).parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4)
    steps = epochs * len(loader)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / 300) * 0.5 * (1 + np.cos(np.pi * s / steps)))
    scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')
    history = []
    with wandb.init(project=PROJECT, name=name, group=name, job_type='train', mode=WANDB_MODE,
                    config=model.cfg | dict(data=data, epochs=epochs, lr=lr, batch=batch, imgsz=IMGSZ)) as run:
        for epoch in range(epochs):
            model.train()
            for s, (imgs, targets) in enumerate(loader):
                imgs = [im.to(DEVICE).float() / 255 for im in imgs]
                targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
                with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                    losses = model(imgs, targets)
                opt.zero_grad()
                scaler.scale(sum(losses.values())).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(params, 10.0)
                scaler.step(opt); scaler.update(); sched.step()
                if s % 50 == 0:
                    run.log({f'train/{k}': v.item() for k, v in losses.items()} | {'lr': sched.get_last_lr()[0]})
            history.append({'epoch': epoch + 1} | {k: v for k, v in evaluate(predict_tv(model, VAL)[0]).items() if k in METRICS})
            run.log({f'val/{k}': v for k, v in history[-1].items()})
            print(f'{name} epoch {epoch + 1}: mAP {history[-1]["mAP"]:.4f}')
    torch.save(model.state_dict(), path)
    pd.DataFrame(history).to_csv(path.parent / 'history.csv', index=False)
    return model.eval()

### Model and experiments

In [ ]:
GIOU_WEIGHT = 10.0

def giou_fastrcnn_loss(class_logits, box_regression, labels, regression_targets):
    labels, targets = torch.cat(labels), torch.cat(regression_targets)
    pos = torch.where(labels > 0)[0]
    pred = box_regression.reshape(len(labels), -1, 4)[pos, labels[pos]]
    unit = torch.tensor([[0., 0., 1., 1.]], device=pred.device).expand(len(pos), 4)
    coder = det_utils.BoxCoder((10., 10., 5., 5.))
    giou = generalized_box_iou_loss(coder.decode_single(pred.float(), unit), coder.decode_single(targets[pos].float(), unit), reduction='sum')
    return F.cross_entropy(class_logits, labels), GIOU_WEIGHT * giou / labels.numel()

def faster_rcnn(giou=False, **kw):
    m = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT', min_size=IMGSZ * 9 // 16, max_size=IMGSZ, box_score_thresh=CONF,
                                   box_nms_thresh=NMS_IOU, box_detections_per_img=MAX_DET, **kw)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, len(CLASSES) + 1)
    m.cfg = dict(kw, giou=giou)
    return m

FRCNN_STEPS = {
    'frcnn_base': {},
    'frcnn_proposals': dict(rpn_pre_nms_top_n_train=4000, rpn_post_nms_top_n_train=4000, rpn_pre_nms_top_n_test=2000, rpn_post_nms_top_n_test=2000),
    'frcnn_sampling': dict(box_positive_fraction=0.5),
    'frcnn_giou': dict(giou=True),
}
best_frcnn, frcnn_cfg = tune(FRCNN_STEPS, lambda name, **c: train_tv(name, faster_rcnn(**c), lr=0.005))
table(FRCNN_STEPS)

### What each stage contributes

In [ ]:
def size_group(o):
    return pd.cut(o.w * o.h, [0, 32 ** 2, 96 ** 2, np.inf], right=False, labels=['small', 'medium', 'large'])

@torch.no_grad()
def stage_recall(model, fr, score=0.05):
    store = {}
    hook = model.rpn.register_forward_hook(lambda m, args, out: store.update(boxes=out[0][0], size=args[0].image_sizes[0]))
    rows, n_boxes = [], []
    for f in fr.itertuples():
        out = model([decode_image(f.path, ImageReadMode.RGB).to(DEVICE).float() / 255])[0]
        g = objs[objs.image_id == f.image_id]
        gt = torch.tensor(xyxy(g[['x', 'y', 'w', 'h']].values), dtype=torch.float32, device=DEVICE)
        props = resize_boxes(store['boxes'], store['size'], (f.height, f.width))
        keep = out['scores'] >= score
        same = out['labels'][keep][None] == torch.tensor(g.cls.values + 1, device=DEVICE)[:, None]
        detected = (box_iou(gt, out['boxes'][keep]) * same).max(1).values >= .5 if keep.any() else torch.zeros(len(g), dtype=torch.bool)
        rows.append(pd.DataFrame({'size': size_group(g), 'proposed': (box_iou(gt, props).max(1).values >= .5).cpu().numpy(), 'detected': detected.cpu().numpy()}))
        n_boxes.append((len(props), int(keep.sum())))
    hook.remove()
    r = pd.concat(rows)
    r['outcome'] = np.select([r.detected, r.proposed], ['detected', 'lost in the second stage'], 'never proposed')
    print('boxes per frame: %.0f proposals -> %.0f detections with score >= %.2f' % (*np.mean(n_boxes, axis=0), score))
    return (pd.crosstab(r['size'], r.outcome, normalize='index', margins=True) * 100).round(1)

frcnn = train_tv(best_frcnn, faster_rcnn(**frcnn_cfg), lr=0.005)
stage_recall(frcnn, VAL.sample(100, random_state=SEED))

### Failure cases

In [ ]:
TYPE_COLORS = {'correct': '#1baf7a', 'missed': '#eda100', 'wrong class': '#eb6834', 'background': '#e34948', 'duplicate': '#6250d6', 'localization': '#e87ba4', 'ignored region': '#888888'}

def label_errors(name, thr=0.3):
    dets = pd.DataFrame(load_dets(name)).query('score >= @thr').sort_values('score', ascending=False)
    dets[['x', 'y', 'w', 'h']] = np.array(dets.bbox.tolist(), dtype=float).reshape(-1, 4)
    dets['type'] = 'background'
    gts = objs[objs.image_id.isin(VAL.image_id)].assign(found=False, pred=8)
    for i, d in dets.groupby('image_id'):
        g = gts[gts.image_id == i]
        if len(g) == 0:
            continue
        iou = box_iou(torch.tensor(xyxy(d[['x', 'y', 'w', 'h']].values)), torch.tensor(xyxy(g[['x', 'y', 'w', 'h']].values))).numpy()
        same = d.category_id.values[:, None] == g.category_id.values[None]
        used, types = np.zeros(len(g), bool), []
        for j in range(len(d)):
            free = iou[j] * same[j] * ~used
            if free.max() >= .5:
                used[free.argmax()] = True
                types.append('correct')
            elif (iou[j] * same[j]).max() >= .5:
                types.append('duplicate')
            elif (iou[j] * ~same[j]).max() >= .5:
                types.append('wrong class')
            else:
                types.append('localization' if iou[j].max() >= .1 else 'background')
        dets.loc[d.index, 'type'] = types
        gts.loc[g.index, 'found'] = used
        gts.loc[g.index, 'pred'] = np.where(iou.max(0) >= .5, d.category_id.values[iou.argmax(0)] - 1, 8)
    dets.loc[(dets.type == 'background') & (covered(dets, ignore) > .5), 'type'] = 'ignored region'
    return dets, gts

def error_plots(name, thr=0.3):
    dets, gts = label_errors(name, thr)
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.5), gridspec_kw={'width_ratios': [1, 1, 1.2]})
    dets.type.value_counts(normalize=True).plot.barh(ax=ax[0], color=[TYPE_COLORS[t] for t in dets.type.value_counts().index], title=f'{name}: detections with score >= {thr}')
    recall = gts.assign(size=size_group(gts)).groupby(['size', 'occlusion'], observed=True).found.mean().unstack()
    recall.rename(columns={0: 'no occlusion', 1: 'partial', 2: 'heavy'}).plot.bar(ax=ax[1], rot=0, color=COLORS[:3], title='recall by COCO size and occlusion')
    conf = pd.crosstab(gts.cls, gts.pred, normalize='index').reindex(index=range(8), columns=range(9), fill_value=0)
    ax[2].imshow(conf, cmap='Blues', vmin=0, vmax=1)
    ax[2].set_xticks(range(9), CLASSES + ['missed'], rotation=45)
    ax[2].set_yticks(range(8), CLASSES)
    ax[2].set(xlabel='predicted', ylabel='true', title='confusion (rows sum to 1)')
    for (r, c), v in np.ndenumerate(conf.values):
        if v >= .02:
            ax[2].text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=7, color='white' if v > .5 else 'black')
    plt.tight_layout(); plt.show()
    return dets, gts

def show_cases(dets, gts, width=640):
    cases = {'missed small objects': gts[~gts.found & (gts.w * gts.h < 32 ** 2)], 'wrong class': dets[dets.type == 'wrong class'], 'background': dets[dets.type == 'background']}
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
    for a, (title, err) in zip(ax, cases.items()):
        if len(err) == 0:
            a.axis('off')
            continue
        i = err.image_id.value_counts().index[0]
        f, e = frames.loc[i], err[err.image_id == i]
        x0 = np.clip((e.x + e.w / 2).median() - width / 2, 0, f.width - width)
        y0 = np.clip((e.y + e.h / 2).median() - width * 9 / 32, 0, f.height - width * 9 / 16)
        d, g = dets[dets.image_id == i], gts[(gts.image_id == i) & ~gts.found]
        draw(a, f.path, [*d[['x', 'y', 'w', 'h']].values, *g[['x', 'y', 'w', 'h']].values], [TYPE_COLORS[t] for t in d.type] + [TYPE_COLORS['missed']] * len(g),
             crop=(x0, y0, x0 + width, y0 + width * 9 / 16), title=f'{title}: {f.key}')
    fig.legend(handles=[plt.Line2D([], [], color=c, label=t) for t, c in TYPE_COLORS.items()], loc='lower center', ncol=7)
    plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

dets, gts = error_plots(best_frcnn)
show_cases(dets, gts)

## 4. One-stage anchor-based: RetinaNet

### Anchors from the data

In [ ]:
fit_objs = objs[~objs.holdout]
log_ratio = np.log(fit_objs.h / fit_objs.w).sample(100_000, random_state=SEED).to_numpy()[:, None]
RATIOS = tuple(np.exp(np.sort(KMeans(3, n_init=10, random_state=SEED).fit(log_ratio).cluster_centers_.ravel())).round(2).tolist())

def anchor_stats(sizes, ratios, fg=0.5, bg=0.4, n=300):
    gen = AnchorGenerator(tuple((s, s * 2 ** (1 / 3), s * 2 ** (2 / 3)) for s in sizes), (tuple(ratios),) * 5)
    H, W = IMGSZ * 9 // 16, IMGSZ
    anchors = gen(ImageList(torch.zeros(1, 3, H, W), [(H, W)]), [torch.zeros(1, 1, -(-H // s), -(-W // s)) for s in [8, 16, 32, 64, 128]])[0]
    matcher = det_utils.Matcher(fg, bg, allow_low_quality_matches=True)
    sample = fit_objs[fit_objs.image_id.isin(frames.image_id[~frames.holdout].sample(n, random_state=SEED))]
    rows = []
    for _, g in sample.groupby('image_id'):
        iou = box_iou(torch.tensor(xyxy(g[['x', 'y', 'w', 'h']].values) * IMGSZ / max(g.width.iloc[0], g.height.iloc[0]), dtype=torch.float32), anchors)
        match = matcher(iou.clone())
        rows.append(pd.DataFrame({'side': g.side_in.values, 'strong': (iou >= fg).sum(1).numpy(), 'positives': torch.bincount(match[match >= 0], minlength=len(g)).numpy()}))
    r = pd.concat(rows)
    groups = pd.cut(r.side, [0, 16, 32, 64, np.inf], labels=['<16px', '16-32px', '32-64px', '>64px'])
    return {'objects with an anchor IoU >= 0.5, %': (r.strong > 0).mean() * 100,
            **r.groupby(groups, observed=True).positives.mean().add_prefix('positives per object ').to_dict(),
            'positive anchors per frame': r.positives.sum() / sample.image_id.nunique(), 'anchors per frame': len(anchors)}

ANCHOR_SETS = {
    'default': ((32, 64, 128, 256, 512), (0.5, 1.0, 2.0)),
    'half': ((16, 32, 64, 128, 256), (0.5, 1.0, 2.0)),
    'quarter': ((8, 16, 32, 64, 128), (0.5, 1.0, 2.0)),
    'quarter, k-means ratios': ((8, 16, 32, 64, 128), RATIOS),
    'half, k-means ratios': ((16, 32, 64, 128, 256), RATIOS),
}
anchor_table = pd.DataFrame({k: anchor_stats(*v) for k, v in ANCHOR_SETS.items()}).T.round(2)
RETINA_ANCHORS = ANCHOR_SETS[anchor_table['objects with an anchor IoU >= 0.5, %'].idxmax()]
print('k-means aspect ratios (h/w):', RATIOS, '| selected anchors:', RETINA_ANCHORS)
anchor_table

### Model and experiments

In [ ]:
NWD_C = fit_objs.side_in.mean()

def nwd_loss(pred, target):
    p, t = box_convert(pred, 'xyxy', 'cxcywh'), box_convert(target, 'xyxy', 'cxcywh')
    w2 = ((p[:, :2] - t[:, :2]) ** 2).sum(1) + (((p[:, 2:] - t[:, 2:]) / 2) ** 2).sum(1)
    return 1 - torch.exp(-torch.sqrt(w2 + 1e-7) / NWD_C)

def box_loss(type, box_coder, anchors, gt_boxes, regression, cnf=None):
    if type != 'nwd':
        return det_utils._box_loss(type, box_coder, anchors, gt_boxes, regression, cnf)
    return nwd_loss(box_coder.decode_single(regression.float(), anchors), gt_boxes).sum()

tv_retinanet._box_loss = box_loss

def retinanet(sizes=(32, 64, 128, 256, 512), ratios=(0.5, 1.0, 2.0), fg=0.5, bg=0.4, loss='giou'):
    m = retinanet_resnet50_fpn_v2(weights='DEFAULT', min_size=IMGSZ * 9 // 16, max_size=IMGSZ, score_thresh=CONF, nms_thresh=NMS_IOU, detections_per_img=MAX_DET)
    head = m.head.classification_head
    head.num_classes = len(CLASSES) + 1
    head.cls_logits = nn.Conv2d(256, head.num_anchors * head.num_classes, 3, padding=1)
    nn.init.normal_(head.cls_logits.weight, std=0.01)
    nn.init.constant_(head.cls_logits.bias, -np.log(99))
    m.anchor_generator = AnchorGenerator(tuple((s, s * 2 ** (1 / 3), s * 2 ** (2 / 3)) for s in sizes), (tuple(ratios),) * 5)
    m.proposal_matcher = det_utils.Matcher(fg, bg, allow_low_quality_matches=True)
    m.head.regression_head._loss_type = loss
    m.cfg = dict(sizes=sizes, ratios=ratios, fg=fg, bg=bg, loss=loss)
    return m

RETINA_STEPS = {
    'retina_base': {},
    'retina_anchors': dict(sizes=RETINA_ANCHORS[0], ratios=RETINA_ANCHORS[1]),
    'retina_iou': dict(fg=0.4, bg=0.3),
    'retina_nwd': dict(loss='nwd'),
}
best_retina, retina_cfg = tune(RETINA_STEPS, lambda name, **c: train_tv(name, retinanet(**c), lr=0.0025))
table([*RETINA_STEPS, best_frcnn])

### Comparison with the two-stage model

In [ ]:
def compare_errors(names, thr=0.3):
    rows = {}
    for n in names:
        dets, gts = label_errors(n, thr)
        rows[n] = gts.groupby(size_group(gts), observed=True).found.mean().add_prefix('recall ').to_dict() | dets.type.value_counts(normalize=True).to_dict()
    return (pd.DataFrame(rows).T * 100).round(1)

compare_errors([best_frcnn, best_retina])

## 5. One-stage anchor-free: YOLO26

### Grid points per object

In [ ]:
wh_in = fit_objs[['w', 'h']].to_numpy() * (fit_objs.side_in / fit_objs.side).to_numpy()[:, None]
pd.DataFrame({f'< 1 cell at {size}, %': {f'stride {s}': ((wh_in[:, 0] * size / IMGSZ / s) * (wh_in[:, 1] * size / IMGSZ / s) < 1).mean() * 100 for s in [4, 8, 16, 32]}
              for size in [IMGSZ, 1280]}).round(1)

### Experiments

In [ ]:
def train_ultra(name, model='yolo26m', p2=False, nq=None, nd=None, data=f'fit{TRAIN_STEP}', imgsz=IMGSZ, epochs=EPOCHS, **kw):
    weights = RUNS / name / 'weights' / 'last.pt'
    Model = RTDETR if model.startswith('rtdetr') else YOLO
    done = weights.exists() and torch.load(weights, map_location='cpu', weights_only=False)['epoch'] == -1
    if not done:
        check_time()
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        per_gpu = 2 if p2 or model == 'rtdetr-x' else 4 if Model is RTDETR or imgsz > IMGSZ else 8
        while True:
            gpus = dict(device=list(range(GPUS)), batch=per_gpu * GPUS) if GPUS > 1 else dict(batch=-1)
            gc.collect(); torch.cuda.empty_cache()
            try:
                if weights.exists():
                    Model(weights).train(resume=True, **gpus)
                else:
                    if p2:
                        net = YOLO(f'{model}-p2.yaml').load(f'{model}.pt')
                    elif nq or nd:
                        net = RTDETR(rtdetr_yaml(model, nq or 300, nd or 100)).load(f'{model}.pt')
                    else:
                        net = Model(f'{model}.pt')
                    net.train(data=str(DS / f'{data}.yaml'), project=PROJECT, name=name, save_dir=str((RUNS / name).resolve()), imgsz=imgsz, epochs=epochs,
                              close_mosaic=3, seed=SEED, workers=4, **gpus, **kw)
                break
            except subprocess.CalledProcessError:
                if GPUS < 2 or per_gpu == 1:
                    raise
                per_gpu //= 2
                print(f'{name}: DDP training failed, retrying with {per_gpu} images per GPU')
    return Model(weights)

YOLO_STEPS = {
    'yolo_base': dict(model='yolo26m'),
    'yolo_1280': dict(imgsz=1280),
    'yolo_p2': dict(p2=True),
}
best_yolo, yolo_cfg = tune(YOLO_STEPS, train_ultra)
table([*YOLO_STEPS, best_frcnn, best_retina])

In [ ]:
compare_errors([best_frcnn, best_retina, best_yolo])

## 6. Query-based: RT-DETR

In [ ]:
def rtdetr_yaml(model='rtdetr-l', nq=300, nd=100):
    cfg = yaml.safe_load(open(Path(ultralytics.__file__).parent / f'cfg/models/rt-detr/{model}.yaml'))
    cfg['head'][-1][-1] = ['nc', 256, nq, 4, 8, 6, 1024, 0.0, None, -1, nd]
    path = RUNS / f'{model}-q{nq}-dn{nd}.yaml'
    yaml.safe_dump(cfg, open(path, 'w'))
    return str(path)

RTDETR_STEPS = {
    'rtdetr_base': dict(model='rtdetr-l'),
    'rtdetr_q500': dict(nq=500),
    'rtdetr_dn500': dict(nd=500),
}
best_rtdetr, rtdetr_cfg = tune(RTDETR_STEPS, train_ultra)
table([*RTDETR_STEPS, best_frcnn, best_retina, best_yolo])

### Validation mAP during training

In [ ]:
def curve(name):
    if (RUNS / name / 'results.csv').exists():
        r = pd.read_csv(RUNS / name / 'results.csv')
        return r.epoch, r['metrics/mAP50-95(B)']
    r = pd.read_csv(RUNS / name / 'history.csv')
    return r.epoch, r.mAP

BEST = [best_frcnn, best_retina, best_yolo, best_rtdetr]
json.dump(dict(zip(['frcnn', 'retina', 'yolo', 'rtdetr'], BEST)), open(RUNS / 'best_runs.json', 'w'))
fig, ax = plt.subplots(figsize=(8, 4))
for n, c in zip(dict.fromkeys(BEST + ['rtdetr_base', 'yolo_base']), COLORS):
    ax.plot(*curve(n), 'o-', color=c, ms=6, lw=2, label=n)
ax.set(xlabel='epoch', ylabel='validation mAP', title='validation mAP during training')
ax.legend(); plt.show()

### Small objects and dense scenes

In [ ]:
n_val = per_frame.loc[VAL.image_id]
density = pd.qcut(n_val, 3, labels=['sparse', 'medium', 'dense'])
print('objects per frame in each group:', {d: f'{n_val[density == d].min()}-{n_val[density == d].max()}' for d in density.cat.categories})
by_density = pd.DataFrame({n: {d: evaluate(load_dets(n), n_val.index[density == d])['mAP'] for d in density.cat.categories} for n in BEST})
pd.concat([table(BEST)[['mAP', 'AP_small', 'AP_medium', 'AP_large', 'ms/img']], by_density.T.add_prefix('mAP ')], axis=1)

In [ ]:
compare_errors(BEST)

## 7. Your Pipeline

### 7.1 Training changes

In [ ]:
if not (RUNS / 'pipe_step5' / 'val_metrics.json').exists():
    prepare(5)
pipe_base, base_cfg = best_rtdetr, rtdetr_cfg
PIPE_STEPS = {
    'pipe_step5': dict(data='fit5'),
    'pipe_long': dict(epochs=2 * EPOCHS),
    'pipe_adamw': dict(optimizer='AdamW', lr0=1e-4, nbs=8, cos_lr=True, warmup_epochs=1),
    'pipe_1280': dict(imgsz=1280),
    'pipe_x': dict(model='rtdetr-x'),
}
best_pipe, pipe_cfg = tune(PIPE_STEPS, train_ultra, cfg=base_cfg, best=pipe_base)

def merge(*det_lists):
    d = pd.DataFrame([x for dets in det_lists for x in dets])
    out = []
    for i, g in d.groupby('image_id'):
        keep = batched_nms(torch.tensor(xyxy(np.stack(g.bbox.values)), dtype=torch.float32), torch.tensor(g.score.values, dtype=torch.float32),
                           torch.tensor(g.category_id.values), NMS_IOU)[:MAX_DET]
        out += g.iloc[keep.numpy()].to_dict('records')
    return out

results['ensemble yolo + rtdetr'] = evaluate(merge(load_dets(best_yolo), load_dets(best_rtdetr))) | {'ms/img': results[best_yolo]['ms/img'] + results[best_rtdetr]['ms/img']}
table([pipe_base, *PIPE_STEPS, 'ensemble yolo + rtdetr'])

### 7.2 Final weights and input size

In [ ]:
def predict_scaled(model, fr, scale=1.0):
    return predict_ultra(model, fr, imgsz=round(model.overrides['imgsz'] * scale / 32) * 32)[0]

def cached(path, fn):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(fn()))
    return json.loads(path.read_text())

def timed_eval(fn):
    t = time.perf_counter()
    dets = fn()
    return evaluate(dets) | {'ms/img': round((time.perf_counter() - t) / len(VAL) * 1000, 1)}

SCALES = {'full frame': 1.0, 'full frame, input x1.25': 1.25, 'full frame, input x1.5': 1.5}
cfgs = {pipe_base: base_cfg, best_pipe: pipe_cfg}
grid = {}
for n, cfg in cfgs.items():
    model = train_ultra(n, **cfg)
    for s, scale in SCALES.items():
        grid[n, s] = cached(RUNS / 'pipeline' / f'{n} {s}.json', lambda: timed_eval(lambda: predict_scaled(model, VAL, scale)))
best_pipe, FINAL_STRATEGY = max(grid, key=lambda k: grid[k]['mAP'])
pipe_cfg = cfgs[best_pipe]
print('final weights:', best_pipe, '| inference:', FINAL_STRATEGY)
pd.DataFrame(grid).T[METRICS + ['ms/img']]

### 7.3 Summary of all models

In [ ]:
display(table(results).sort_values('mAP', ascending=False))
fig, ax = plt.subplots(figsize=(7, 4))
for n, c in zip(dict.fromkeys(BEST + [best_pipe]), COLORS):
    ax.scatter(results[n]['ms/img'], results[n]['mAP'], s=60, color=c, label=n)
ax.set(xlabel='ms per frame (fp32, batch 1)', ylabel='validation mAP', title='accuracy vs speed')
ax.legend(); plt.show()
pd.DataFrame({n: {c: results[n][c] for c in CLASSES} for n in BEST + [best_pipe]}).T.round(3)

### Pipeline results

## 8. Final model and submission

In [ ]:
final_cfg = pipe_cfg | {'data': pipe_cfg.get('data', f'fit{TRAIN_STEP}').replace('fit', 'all')}
setup = {'fit_run': best_pipe, 'fit_cfg': pipe_cfg, 'final_cfg': final_cfg, 'strategy': FINAL_STRATEGY}
old = json.load(open(RUNS / 'final_setup.json')) if (RUNS / 'final_setup.json').exists() else {}
if old.get('final_cfg') != final_cfg:
    shutil.rmtree(RUNS / 'final', ignore_errors=True)
elif old != setup:
    for f in ['test_preds.json', 'holdout_preds.json']:
        (RUNS / 'final' / f).unlink(missing_ok=True)
json.dump(setup, open(RUNS / 'final_setup.json', 'w'))
final = train_ultra('final', **final_cfg, val=False)
test_fr = test.assign(image_id=test.index, path=[str(DATA / f) for f in test.file_name])
test_dets = cached(RUNS / 'final' / 'test_preds.json', lambda: predict_scaled(final, test_fr, SCALES[FINAL_STRATEGY]))
cached(RUNS / 'final' / 'holdout_preds.json', lambda: predict_scaled(train_ultra(best_pipe, **pipe_cfg), frames[frames.holdout], SCALES[FINAL_STRATEGY]))
json.dump(setup, open(RUNS / 'final' / 'final.json', 'w'))

def submission(dets, ids):
    d = pd.DataFrame(dets).sort_values('score', ascending=False).groupby('image_id').head(MAX_DET)
    d['text'] = [f'{c - 1} {s:.4f} {x:.1f} {y:.1f} {w:.1f} {h:.1f}' for c, s, (x, y, w, h) in zip(d.category_id, d.score, d.bbox)]
    sub = pd.DataFrame({'image_id': ids, 'PredictionString': d.groupby('image_id').text.agg(' '.join).reindex(range(len(ids))).fillna('').values})
    sample_file = DATA / 'sample_submission.csv'
    return sub.set_index('image_id').loc[pd.read_csv(sample_file).image_id].reset_index() if sample_file.exists() else sub

sub = submission(test_dets, test.image_id)
sub.to_csv('submission.csv', index=False)
print(sub.shape, '| detections per frame: %.0f' % (sub.PredictionString.str.split().str.len().fillna(0) / 6).mean())
sub.head(3)

## Bonus tasks

### B1. Data temporality

#### Track linking and rescoring

In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from torchvision.ops import roi_align

BONUS = RUNS / 'bonus'
BONUS.mkdir(exist_ok=True)
BOX = ['x1', 'y1', 'x2', 'y2']

def wb_log(name, **items):
    with wandb.init(project=PROJECT, id=name, name=name, group='bonus', job_type='bonus', resume='allow', mode=WANDB_MODE) as run:
        run.log({k: wandb.Image(v) if isinstance(v, plt.Figure) else wandb.Table(dataframe=v.reset_index().rename(columns=str)) for k, v in items.items()})

def link(dets, meta, thr=0.05, iou_min=0.4, max_gap=2):
    d = pd.DataFrame(dets).merge(meta[['image_id', 'video', 'frame']], on='image_id', how='left')
    b = np.array(d.bbox.tolist(), dtype=float).reshape(-1, 4)
    d[BOX] = np.c_[b[:, :2], b[:, :2] + b[:, 2:]]
    d = d.drop(columns='bbox').join(meta.groupby('video').frame.agg(fmin='min', fmax='max'), on='video')
    c = np.flatnonzero(d.score.to_numpy() >= thr)
    key = np.c_[pd.factorize(d.video)[0], d.category_id.to_numpy(), d.frame.to_numpy()][c]
    o = np.lexsort(key.T[::-1])
    c, key = c[o], key[o]
    cuts = np.r_[0, np.flatnonzero((np.diff(key, axis=0) != 0).any(1)) + 1, len(c)]
    xy = d[BOX].to_numpy()
    track, nxt, group = np.full(len(d), -1), 0, None
    for s, e in zip(cuts[:-1], cuts[1:]):
        rows, f = c[s:e], key[s, 2]
        if group != tuple(key[s, :2]):
            group, last_box, last_f, last_id = tuple(key[s, :2]), np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)
        ids = np.full(len(rows), -1)
        alive = f - last_f <= max_gap + 1
        if alive.any():
            a = np.flatnonzero(alive)
            iou = box_iou(torch.from_numpy(last_box[a]), torch.from_numpy(xy[rows])).numpy()
            r, k = linear_sum_assignment(-iou)
            ok = iou[r, k] >= iou_min
            ids[k[ok]] = last_id[a[r[ok]]]
        new = ids < 0
        ids[new] = np.arange(nxt, nxt + new.sum())
        nxt += new.sum()
        track[rows] = ids
        keep = alive & ~np.isin(last_id, ids)
        last_box, last_f, last_id = np.r_[last_box[keep], xy[rows]], np.r_[last_f[keep], np.full(len(rows), f)], np.r_[last_id[keep], ids]
    free = track < 0
    track[free] = nxt + np.arange(free.sum())
    return d.assign(track=track)

def gaps(d, fill, max_gap):
    s = d[d.track.duplicated(keep=False)].sort_values(['track', 'frame'])
    n = s[['track', 'frame', 'score', *BOX]].shift(-1)
    step = (n.frame - s.frame).to_numpy()
    m = (n.track == s.track).to_numpy() & (step > 1) & (step <= max_gap + 1)
    a, b, step = s[m], n[m], step[m]
    parts = []
    for k in range(1, max_gap + 1):
        sel = step > k
        t = (k / step[sel])[:, None]
        box = a[BOX].to_numpy()[sel] * (1 - t) + b[BOX].to_numpy()[sel] * t
        parts.append(a[sel][['video', 'category_id', 'track', 'fmin', 'fmax']].assign(
            frame=a.frame.to_numpy()[sel] + k, score=fill * np.minimum(a.score.to_numpy(), b.score.to_numpy())[sel], **dict(zip(BOX, box.T))))
    return pd.concat(parts).merge(d[['video', 'frame', 'image_id']].drop_duplicates(), on=['video', 'frame'])

def rescore(d, alpha=0.5, W=5, causal=False, fill=0.0, max_gap=2):
    t, f = d.track.to_numpy().astype(np.int64), d.frame.to_numpy().astype(np.int64)
    key = t * 10 ** 7 + f
    o = np.argsort(key, kind='stable')
    cs = np.r_[0, np.cumsum(d.score.to_numpy()[o])]
    lo = np.searchsorted(key[o], key - W, 'left')
    hi = np.searchsorted(key[o], key if causal else key + W, 'right')
    slots = np.minimum(f if causal else f + W, d.fmax.to_numpy()) - np.maximum(f - W, d.fmin.to_numpy()) + 1
    out = d.assign(score=alpha * d.score.to_numpy() + (1 - alpha) * (cs[hi] - cs[lo]) / slots)
    if fill:
        out = pd.concat([out, gaps(out, fill, max_gap)], ignore_index=True)
    return out[out.groupby('image_id').score.rank(ascending=False, method='first') <= MAX_DET]

def to_dets(d):
    return [{'image_id': i, 'category_id': c, 'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': s}
            for i, c, x1, y1, x2, y2, s in zip(*(d[k].tolist() for k in ['image_id', 'category_id', *BOX, 'score']))]

#### Tuning on one half of the hold-out clips, testing on the other

In [ ]:
hold = frames[frames.holdout]
val_ids = set(VAL.image_id)
t0 = time.perf_counter()
T = link(json.load(open(RUNS / 'final' / 'holdout_preds.json')), hold)
link_ms = (time.perf_counter() - t0) / len(hold) * 1000
lengths = T[T.score >= .05].track.value_counts()
print(f'{len(hold)} hold-out frames | {len(lengths)} tracks from detections with score >= 0.05 | tracks longer than 1 frame: {(lengths > 1).mean():.0%} | median length {lengths[lengths > 1].median():.0f} frames')

order = VAL.video.value_counts().index
HALVES = {'A': order[0::2], 'B': order[1::2]}
IDS = {h: VAL.image_id[VAL.video.isin(c)].tolist() for h, c in HALVES.items()}
MODES = {'causal': dict(causal=True), 'bidirectional': dict(causal=False), 'bidirectional + gap fill': dict(causal=False, fill=0.5)}
CONFIGS = [dict(mode=m, alpha=a, W=w) for m in MODES for a in (0.3, 0.5, 0.7) for w in (2, 5, 10)]

def run_cfg(c, d):
    return rescore(d, alpha=c['alpha'], W=c['W'], **MODES[c['mode']])

def score_on(d, ids):
    return evaluate(to_dets(d[d.image_id.isin(ids)]), ids)

def grid_row(c):
    d = run_cfg(c, T)
    return c | {h: score_on(d, IDS[h])['mAP'] for h in IDS}

base = cached(BONUS / 'temporal_base.json', lambda: {h: score_on(T, IDS[h])['mAP'] for h in IDS})
tgrid = pd.DataFrame(cached(BONUS / 'temporal_grid.json', lambda: [grid_row(c) for c in CONFIGS]))
tgrid['mean'] = tgrid[['A', 'B']].mean(axis=1)
cross = pd.DataFrame({f'tuned on {a}, tested on {b}': {'clips': len(HALVES[b]), 'frames': len(IDS[b]), **tgrid.loc[tgrid[a].idxmax(), ['mode', 'alpha', 'W']].to_dict(),
                                                       'baseline mAP': base[b], 'temporal mAP': tgrid.loc[tgrid[a].idxmax(), b]} for a, b in [('A', 'B'), ('B', 'A')]}).T
TEMPORAL = json.loads(tgrid.loc[tgrid['mean'].idxmax(), ['mode', 'alpha', 'W']].to_json())
CAUSAL = json.loads(tgrid[tgrid['mode'] == 'causal'].sort_values('mean').iloc[-1][['mode', 'alpha', 'W']].to_json())
pivot = tgrid.pivot_table(index=['mode', 'W'], columns='alpha', values='mean').round(4)
wb_log('bonus_temporal', grid=pivot, cross_fit=cross)
print('baseline mAP per half:', base, '| selected on both halves:', TEMPORAL)
display(pivot)
cross

#### Effect on the validation frames

In [ ]:
t0 = time.perf_counter()
variants = {'final_fit': T, 'final_causal': run_cfg(CAUSAL, T), 'final_temporal': run_cfg(TEMPORAL, T)}
rescore_ms = (time.perf_counter() - t0) / 2 / len(hold) * 1000
for name, d in variants.items():
    (RUNS / name).mkdir(exist_ok=True)
    dets = to_dets(d[d.image_id.isin(val_ids)])
    json.dump(dets, open(RUNS / name / 'val_preds.json', 'w'))
    results[name] = evaluate(dets) | {'ms/img': round(grid[best_pipe, FINAL_STRATEGY]['ms/img'] + (name != 'final_fit') * (link_ms + rescore_ms), 1)}

per_class = pd.DataFrame({n: {c: results[n][c] for c in CLASSES} for n in variants}).T
errors = compare_errors(list(variants))
wb_log('bonus_temporal', metrics=table(list(variants)), per_class=per_class, errors=errors)
print(f'temporal step: {link_ms:.1f} ms/frame linking + {rescore_ms:.1f} ms/frame rescoring on CPU')
display(table(list(variants)))
display(per_class)
errors

#### Which detections are boosted and which are suppressed

In [ ]:
old, _ = label_errors('final_fit', thr=0.05)
rows = T.index[T.image_id.isin(val_ids)][old.index]
new = variants['final_temporal'].score.reindex(rows).fillna(0).to_numpy()
old['log2 change'] = np.log2(np.maximum(new, 1e-4) / old.score.to_numpy())
old['track length'] = pd.cut(T.track.map(T.track.value_counts()).loc[rows].to_numpy(), [0, 1, 5, 20, 100, np.inf], labels=['1', '2-5', '6-20', '21-100', '>100'])
types = [t for t in ['correct', 'localization', 'duplicate', 'wrong class', 'ignored region', 'background'] if t in set(old.type)]

stats = T[T.score >= .05].groupby('track').agg(n=('frame', 'size'), med=('score', 'median'))
pick = stats[stats.med.between(.15, .5)].nlargest(3, 'n').index
fig, ax = plt.subplots(1, 3, figsize=(18, 4.2), gridspec_kw={'width_ratios': [1.2, 1, 1.4]})
ax[0].boxplot([old['log2 change'][old.type == t] for t in types], showfliers=False)
ax[0].set_xticks(range(1, len(types) + 1), types, rotation=30)
ax[0].set(ylabel='log2(new score / old score)', title='score change by error type (score >= 0.05)')
lens = old['track length'].cat.categories
ax[1].boxplot([old['log2 change'][old['track length'] == l] for l in lens], showfliers=False)
ax[1].set_xticks(range(1, len(lens) + 1), lens)
ax[1].set(xlabel='track length, frames', title='score change by track length')
for a in ax[:2]:
    a.axhline(0, color='gray', lw=1)
for tr, c in zip(pick, COLORS):
    tt = T[T.track == tr].sort_values('frame')
    ax[2].plot(tt.frame, tt.score, color=c, lw=0.8, alpha=0.4)
    ax[2].plot(tt.frame, variants['final_temporal'].score.reindex(tt.index), color=c, lw=2, label=f'track {tr} ({CLASSES[tt.category_id.iloc[0] - 1]})')
ax[2].set(xlabel='frame', ylabel='score', title='uncertain tracks: before (thin) and after (thick)')
ax[2].legend(fontsize=8)
plt.tight_layout()
wb_log('bonus_temporal', score_change=fig)
plt.show()
old.groupby('type').agg(detections=('score', 'size'), median_log2_change=('log2 change', 'median'), boosted=('log2 change', lambda s: (s > 0).mean())).loc[types].round(3)

#### A frame where temporal rescoring recovers objects

In [ ]:
o_d, o_g = label_errors('final_fit')
n_d, n_g = label_errors('final_temporal')
recovered, lost = n_g.found & ~o_g.found, o_g.found & ~n_g.found
print(f'objects found at score >= 0.3: {o_g.found.sum()} -> {n_g.found.sum()} | recovered {recovered.sum()}, lost {lost.sum()} | background detections: {(o_d.type == "background").sum()} -> {(n_d.type == "background").sum()}')
if recovered.any():
    i = n_g[recovered].image_id.value_counts().index[0]
    f, r = frames.loc[i], n_g[recovered & (n_g.image_id == i)]
    cw, ch = min(640, f.width), min(360, f.height)
    x0 = np.clip((r.x + r.w / 2).median() - cw / 2, 0, f.width - cw)
    y0 = np.clip((r.y + r.h / 2).median() - ch / 2, 0, f.height - ch)
    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    for a, (title, d, g) in zip(ax, [('baseline', o_d, o_g), ('temporal rescoring', n_d, n_g)]):
        d, g = d[d.image_id == i], g[(g.image_id == i) & ~g.found]
        draw(a, f.path, [*d[['x', 'y', 'w', 'h']].values, *g[['x', 'y', 'w', 'h']].values], [TYPE_COLORS[t] for t in d.type] + [TYPE_COLORS['missed']] * len(g),
             crop=(x0, y0, x0 + cw, y0 + ch), title=f'{title}, {f.key}: {(d.type == "correct").sum()} correct, {len(g)} missed, {(d.type == "background").sum()} background')
    fig.legend(handles=[plt.Line2D([], [], color=c, label=t) for t, c in TYPE_COLORS.items()], loc='lower center', ncol=7)
    plt.tight_layout(rect=(0, 0.06, 1, 1))
    wb_log('bonus_temporal', recovered_frame=fig)
    plt.show()

#### Temporal submission

In [ ]:
print('test clips with missing frame numbers:', int((test.groupby('clip').frame.apply(lambda s: s.sort_values().diff().max()) > 1).sum()))
TT = link(test_dets, test_fr.rename(columns={'clip': 'video'}))
sub_t = submission(to_dets(run_cfg(TEMPORAL, TT)), test.image_id)
sub_t.to_csv('submission_temporal.csv', index=False)
json.dump(TEMPORAL, open(BONUS / 'temporal_config.json', 'w'))
print('submission_temporal.csv', sub_t.shape, '| config:', TEMPORAL)

### What temporal rescoring gives

### B2. Visualization

#### RoI features of ground-truth objects (Faster R-CNN)

In [ ]:
frcnn = train_tv(best_frcnn, faster_rcnn(**frcnn_cfg), lr=0.005)
_, frcnn_gts = label_errors(best_frcnn)
OUTCOME_COLORS = {'detected': '#1baf7a', 'missed, head right': '#eda100', 'missed, head wrong': '#e34948'}

@torch.no_grad()
def roi_embed(model, fr):
    model.eval()
    feats, rows = [], []
    for f in fr.itertuples():
        g = objs[objs.image_id == f.image_id]
        if len(g) == 0:
            continue
        il, _ = model.transform([decode_image(f.path, ImageReadMode.RGB).to(DEVICE).float() / 255])
        box = resize_boxes(torch.tensor(xyxy(g[['x', 'y', 'w', 'h']].values), dtype=torch.float32, device=DEVICE), (f.height, f.width), il.image_sizes[0])
        x = model.roi_heads.box_head(model.roi_heads.box_roi_pool(model.backbone(il.tensors), [box], il.image_sizes))
        p = model.roi_heads.box_predictor(x)[0].softmax(1).cpu().numpy()
        feats.append(x.cpu().numpy())
        rows.append(g.assign(pred=p.argmax(1) - 1, p_true=p[np.arange(len(g)), g.cls.values + 1]))
    return np.concatenate(feats), pd.concat(rows)

emb_X, emb = roi_embed(frcnn, VAL.sample(120, random_state=SEED))
emb['found'] = frcnn_gts.found.reindex(emb.index).fillna(False).astype(bool).to_numpy()
emb['size'] = size_group(emb)
emb['outcome'] = np.select([emb.found, emb.pred == emb.cls], ['detected', 'missed, head right'], 'missed, head wrong')
emb_2d = TSNE(2, perplexity=30, init='pca', random_state=SEED).fit_transform(PCA(50, random_state=SEED).fit_transform(emb_X))

fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
for k, c in enumerate(CLASSES):
    m = emb.cls.to_numpy() == k
    ax[0].scatter(*emb_2d[m].T, s=3, color=COLORS[k], label=c)
sc = ax[1].scatter(*emb_2d.T, s=3, c=np.log2(emb.side_in), cmap='viridis')
fig.colorbar(sc, ax=ax[1], label='log2 size at long side 1024, px')
for o, c in OUTCOME_COLORS.items():
    m = emb.outcome.to_numpy() == o
    ax[2].scatter(*emb_2d[m].T, s=3, color=c, label=f'{o} ({m.mean():.0%})')
for a, t in zip(ax, ['true class', 'object size', 'detector outcome']):
    a.set(xticks=[], yticks=[], title=f't-SNE of RoI features: {t}')
ax[0].legend(markerscale=4, fontsize=8)
ax[2].legend(markerscale=4, fontsize=8)
plt.tight_layout()
wb_log('bonus_viz', roi_tsne=fig)
plt.show()
print(f'{len(emb)} objects from 120 validation frames')

In [ ]:
def heat(ax, m, xlabels, title):
    ax.imshow(m, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(m.shape[1]), xlabels, rotation=45)
    ax.set_yticks(range(8), CLASSES)
    ax.set(ylabel='true class', title=title)
    for (r, c), v in np.ndenumerate(m.values):
        if v >= .02:
            ax.text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=7, color='white' if v > .5 else 'black')

cls = emb.cls.to_numpy()
nbrs = NearestNeighbors(n_neighbors=11, metric='cosine').fit(emb_X).kneighbors(emb_X, return_distance=False)[:, 1:]
mix = pd.crosstab(np.repeat(cls, 10), cls[nbrs].ravel(), normalize='index').reindex(index=range(8), columns=range(8), fill_value=0)
head = pd.crosstab(emb.cls, emb.pred, normalize='index').reindex(index=range(8), columns=[*range(8), -1], fill_value=0)
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
heat(ax[0], mix, CLASSES, 'classes of the 10 nearest neighbours in RoI feature space')
heat(ax[1], head, CLASSES + ['background'], 'RoI head prediction given the ground-truth box')
ax[1].set_xlabel('predicted')
plt.tight_layout()
by_size = emb.assign(head_right=emb.pred == emb.cls).groupby('size', observed=True).agg(objects=('cls', 'size'), head_accuracy=('head_right', 'mean'),
                                                                                       detector_recall=('found', 'mean'), median_p_true=('p_true', 'median')).round(3)
wb_log('bonus_viz', roi_neighbours=fig, head_by_size=by_size)
plt.show()
by_size

#### Grad-CAM on the backbone stages (Faster R-CNN)

In [ ]:
LAYERS = {'layer1': 4, 'layer2': 8, 'layer3': 16, 'layer4': 32}
STAGES = {n: f'C{int(n[-1]) + 1} (stride {s})' for n, s in LAYERS.items()}

def det_types(boxes, labels, g):
    if len(g) == 0:
        return np.full(len(boxes), 'background')
    iou = box_iou(torch.tensor(boxes, dtype=torch.float32), torch.tensor(xyxy(g[['x', 'y', 'w', 'h']].values), dtype=torch.float32)).numpy()
    same = labels[:, None] == g.cls.values[None]
    return np.select([(iou * same).max(1) >= .5, (iou * ~same).max(1) >= .5, iou.max(1) >= .1], ['correct', 'wrong class', 'localization'], 'background')

def gradcam(model, fr, per_frame=6, thr=0.3, ctx=3, size=96):
    model.eval()
    acts, recs = {}, []
    hooks = [getattr(model.backbone.body, n).register_forward_hook(lambda m, i, o, n=n: acts.update({n: o})) for n in LAYERS]
    for f in fr.itertuples():
        img = decode_image(f.path, ImageReadMode.RGB).to(DEVICE).float() / 255
        H, W = model.transform([img])[0].image_sizes[0]
        scale = np.array([W / f.width, H / f.height] * 2)
        out = model([img.requires_grad_()])[0]
        keep = torch.where(out['scores'] >= thr)[0].cpu()
        if len(keep) == 0:
            continue
        keep = keep[torch.randperm(len(keep), generator=torch.Generator().manual_seed(int(f.image_id)))[:per_frame]]
        boxes, labels = out['boxes'][keep].detach().cpu().numpy(), out['labels'][keep].cpu().numpy() - 1
        for k, box, c, t in zip(keep.tolist(), boxes, labels, det_types(boxes, labels, objs[objs.image_id == f.image_id])):
            grads = torch.autograd.grad(out['scores'][k], [acts[n] for n in LAYERS], retain_graph=True)
            (x1, y1, x2, y2), side = box, ctx * max(box[2] - box[0], box[3] - box[1])
            reg = np.array([max((x1 + x2 - side) / 2, 0), max((y1 + y2 - side) / 2, 0), min((x1 + x2 + side) / 2, f.width), min((y1 + y2 + side) / 2, f.height)])
            bx = np.clip(np.round((box - reg[[0, 1, 0, 1]]) / (reg[[2, 3, 2, 3]] - reg[[0, 1, 0, 1]]) * size).astype(int), 0, size)
            rec = dict(image_id=f.image_id, score=out['scores'][k].item(), cls=int(c), type=t, x=x1, y=y1, w=x2 - x1, h=y2 - y1, region=reg,
                       uniform=(x2 - x1) * (y2 - y1) / np.prod(reg[2:] - reg[:2]))
            for (n, s), gr in zip(LAYERS.items(), grads):
                cam = F.relu((gr.mean((2, 3), keepdim=True) * acts[n]).sum(1, keepdim=True)).detach()
                crop = roi_align(cam, [torch.tensor(reg * scale, dtype=torch.float32, device=DEVICE)[None]], size, 1 / s, aligned=True)[0, 0].cpu().numpy()
                rec[n] = crop
                rec[f'share {n}'] = crop[bx[1]:max(bx[3], bx[1] + 1), bx[0]:max(bx[2], bx[0] + 1)].sum() / max(crop.sum(), 1e-12)
            recs.append(rec)
        del out
        acts.clear()
    for h in hooks:
        h.remove()
    return pd.DataFrame(recs)

cams = gradcam(frcnn, VAL.sample(40, random_state=SEED + 1))
cams['size'] = size_group(cams)
cams['area'] = cams.w * cams.h
cams.loc[(cams.type == 'background') & (covered(cams, ignore) > .5), 'type'] = 'ignored region'
pd.crosstab(cams.type, cams['size'], margins=True)

In [ ]:
def pick(mask, by, largest=True):
    s = cams[mask]
    return None if s.empty else (s.nlargest(1, by) if largest else s.nsmallest(1, by)).iloc[0]

examples = {'correct, small': pick(cams.type == 'correct', 'area', False), 'correct, large': pick(cams.type == 'correct', 'area'),
            'background': pick(cams.type == 'background', 'score'), 'wrong class': pick(cams.type == 'wrong class', 'score')}
examples = {k: v for k, v in examples.items() if v is not None}
fig, ax = plt.subplots(len(examples), 5, figsize=(15, 3.2 * len(examples)), squeeze=False)
for row, (title, r) in zip(ax, examples.items()):
    crop = Image.open(frames.loc[r.image_id].path).convert('L').crop(tuple(int(round(v)) for v in r.region)).resize((96, 96))
    sx, sy = 96 / (r.region[2] - r.region[0]), 96 / (r.region[3] - r.region[1])
    for a, n in zip(row, [None, *LAYERS]):
        a.imshow(crop, cmap='gray')
        if n:
            a.imshow(r[n] / max(r[n].max(), 1e-12), cmap='jet', alpha=0.5)
        a.add_patch(plt.Rectangle(((r.x - r.region[0]) * sx, (r.y - r.region[1]) * sy), r.w * sx, r.h * sy, fill=False, ec='white', lw=1))
        a.set_title(f'{STAGES[n]}: in-box {r[f"share {n}"]:.0%}' if n else f'{title}: {CLASSES[r.cls]} {r.score:.2f}', fontsize=9)
        a.axis('off')
plt.tight_layout()
wb_log('bonus_viz', gradcam_examples=fig)
plt.show()

In [ ]:
long = cams.melt(id_vars=['size', 'type'], value_vars=[f'share {n}' for n in LAYERS], var_name='stage', value_name='share')
long['stage'] = long.stage.str[6:].map(STAGES)
shares = {col: long.groupby([col, 'stage'], observed=True).share.mean().unstack()[list(STAGES.values())] for col in ['size', 'type']}
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
for a, (col, m) in zip(ax, shares.items()):
    for (name, v), c in zip(m.iterrows(), COLORS):
        a.plot(range(4), v.values, 'o-', color=c, label=f'{name} ({(cams[col] == name).sum()})')
    a.axhline(cams.uniform.mean(), ls='--', color='gray', label='uniform map')
    a.set_xticks(range(4), list(STAGES.values()))
    a.set(ylabel='share of Grad-CAM mass inside the box', title=f'Grad-CAM mass inside the detected box, by {col}')
    a.legend(fontsize=8)
plt.tight_layout()
summary = pd.concat(shares, names=['group', 'value']).round(3)
wb_log('bonus_viz', gradcam_shares=fig, gradcam_table=summary)
plt.show()
summary

### What the visualizations show

## 9. Final submission candidates

#### Test-time augmentation, YOLO ensemble and temporal rescoring, chosen on the hold-out

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ensemble-boxes==1.0.9'], check=True)
from ensemble_boxes import weighted_boxes_fusion

CAND = RUNS / 'candidates'
CAND.mkdir(exist_ok=True)

def predict_view(model, fr, scale=1.0, flip=False):
    imgsz = round(model.overrides['imgsz'] * scale / 32) * 32
    dets = []
    for i, path, w in zip(fr.image_id, fr.path, fr.width):
        src = np.ascontiguousarray(np.array(Image.open(path).convert('RGB'))[:, ::-1, ::-1]) if flip else path
        r = model.predict(src, conf=CONF, iou=NMS_IOU, max_det=MAX_DET, imgsz=imgsz, verbose=False)[0]
        b = r.boxes.xyxy.cpu().numpy()
        if flip:
            b = np.c_[w - b[:, 2], b[:, 1], w - b[:, 0], b[:, 3]]
        dets += to_coco(i, b, r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy())
    return dets

def wbf(views, fr, weights=None, iou_thr=0.6, skip=0.005):
    groups = [dict(list(pd.DataFrame(v).groupby('image_id'))) for v in views]
    out = []
    for i, w, h in zip(fr.image_id, fr.width, fr.height):
        parts = [g.get(i) for g in groups]
        b, s, l = weighted_boxes_fusion([(xyxy(np.stack(p.bbox.values)) / [w, h, w, h]).clip(0, 1) if p is not None else np.zeros((0, 4)) for p in parts],
                                        [p.score.values if p is not None else np.zeros(0) for p in parts],
                                        [p.category_id.values if p is not None else np.zeros(0) for p in parts], weights=weights, iou_thr=iou_thr, skip_box_thr=skip)
        k = np.argsort(-s)[:MAX_DET]
        out += to_coco(i, b[k] * [w, h, w, h], s[k], l[k].astype(int) - 1)
    return out

def compose(spec, rv, yv, fr):
    r, y = [rv[k] for k in spec['r']], [yv[f] for f in spec.get('y', [])]
    if not y:
        return r[0] if len(r) == 1 else wbf(r, fr)
    if spec['fuse'] == 'nms':
        return merge(r[0] if len(r) == 1 else wbf(r, fr), *y)
    return wbf(r + y, fr, weights=[1 / len(r)] * len(r) + [spec['wy'] / len(y)] * len(y))

def final_candidates(tag, fit_name, final_model, primary, hold_all=None, test_primary=None):
    R, Y = RTDETR(RUNS / fit_name / 'weights' / 'last.pt'), train_ultra(best_yolo, **yolo_cfg)
    if hold_all is None:
        hold_all = cached(CAND / f'{fit_name} {primary} holdout.json', lambda: predict_view(R, hold, SCALES[primary]))
    rv = {(s, f): cached(CAND / f'{fit_name} {s} {f} val.json', lambda: predict_view(R, VAL, SCALES[s], f)) for s in SCALES for f in (False, True) if (s, f) != (primary, False)}
    rv[primary, False] = [d for d in hold_all if d['image_id'] in val_ids]
    yv = {f: cached(CAND / f'{best_yolo} {f} val.json', lambda: predict_view(Y, VAL, 1.0, f)) for f in (False, True)}
    specs, res = {'single': dict(r=[(primary, False)]), 'multi-scale': dict(r=[(s, False) for s in SCALES]), 'multi-scale + flip': dict(r=sorted(rv))}, {}

    def score(name, views=rv):
        res[name] = cached(CAND / f'{tag} {name}.json', lambda: evaluate(compose(specs[name], views, yv, VAL)) | {'views': len(specs[name]['r']) + len(specs[name].get('y', []))})

    for n in list(specs):
        score(n)
    best = max(res, key=lambda n: res[n]['mAP'])
    for n, extra in {'nms': dict(fuse='nms'), 'wbf 1:1': dict(fuse='wbf', wy=1.0), 'wbf 2:1': dict(fuse='wbf', wy=0.5)}.items():
        specs[f'{best} + yolo {n}'] = specs[best] | dict(y=[False, True] if 'flip' in best else [False]) | extra
        score(f'{best} + yolo {n}')
    best = max(res, key=lambda n: res[n]['mAP'])
    tracked = run_cfg(TEMPORAL, link(hold_all, hold))
    specs[f'{best} + temporal'] = specs[best]
    score(f'{best} + temporal', rv | {(primary, False): to_dets(tracked[tracked.image_id.isin(val_ids)])})
    use_t = res[f'{best} + temporal']['mAP'] > res[best]['mAP']
    chosen = f'{best} + temporal' if use_t else best
    build_submission(tag, specs[best], use_t, final_model, primary, test_primary)
    out = pd.DataFrame(res).T[METRICS + ['views']].assign(chosen=lambda t: t.index == chosen)
    wb_log('final_candidates', **{f'{tag}_candidates': out})
    print(f'{tag}: chosen on the hold-out -> {chosen} ({res[chosen]["mAP"]:.4f}) | written submission_{tag}.csv')
    return out

def build_submission(tag, spec, use_t, final_model, primary, test_primary=None):
    rt = {k: test_primary if k == (primary, False) and test_primary is not None else cached(CAND / f'{tag} {k[0]} {k[1]} test.json', lambda: predict_view(final_model, test_fr, SCALES[k[0]], k[1])) for k in spec['r']}
    if use_t:
        rt[primary, False] = to_dets(run_cfg(TEMPORAL, link(rt[primary, False], test_fr.rename(columns={'clip': 'video'}))))
    yt = {}
    if spec.get('y'):
        Yf = train_ultra('yolo_final', **yolo_cfg, data=f'all{TRAIN_STEP}', val=False)
        yt = {f: cached(CAND / f'yolo_final {f} test.json', lambda: predict_view(Yf, test_fr, 1.0, f)) for f in spec['y']}
    submission(compose(spec, rt, yt, test_fr), test.image_id).to_csv(f'submission_{tag}.csv', index=False)

In [ ]:
pipeline_candidates = final_candidates('pipeline', best_pipe, final, FINAL_STRATEGY, json.load(open(RUNS / 'final' / 'holdout_preds.json')), test_dets)
pipeline_candidates

### Which submissions to select

## Bonus tasks (continued)

### B3. Custom architectural decisions

#### Building blocks: RT-DETR-l variants with warm-started weights

In [ ]:
from ultralytics.nn.tasks import RTDETRDetectionModel
from ultralytics.utils.torch_utils import get_flops

ARCH = RUNS / 'arch'
ARCH.mkdir(exist_ok=True)
BASE_YAML = yaml.safe_load(open(Path(ultralytics.__file__).parent / 'cfg/models/rt-detr/rtdetr-l.yaml'))
P2_NECK = [[-1, 1, 'Conv', [256, 1, 1]], [-1, 1, 'nn.Upsample', [None, 2, 'nearest']], [1, 1, 'Conv', [256, 1, 1, None, 1, 1, False]], [[-2, -1], 1, 'Concat', [1]],
           [-1, 3, 'RepC3', [256]], [-1, 1, 'Conv', [256, 3, 2]], [[-1, 22], 1, 'Concat', [1]], [-1, 3, 'RepC3', [256]],
           [-1, 1, 'Conv', [256, 3, 2]], [[-1, 17], 1, 'Concat', [1]], [-1, 3, 'RepC3', [256]]]
P5_TAIL = [[-1, 1, 'Conv', [256, 3, 2]], [[-1, 12], 1, 'Concat', [1]], [-1, 3, 'RepC3', [256]]]
P2_WARM = {22: 17, 26: 21, 27: 22, 29: 24, 30: 22, 32: 24}
PYRAMIDS = {'p3p5': (BASE_YAML['head'][12:18], [21, 24, 27], {i: i for i in range(22, 28)}),
            'p2p4': (P2_NECK, [26, 29, 32], P2_WARM),
            'p2p5': (P2_NECK + P5_TAIL, [26, 29, 32, 35], P2_WARM | {33: 25, 35: 27})}

def build_arch(pyramid='p3p5', ndl=6, nq=500, nd=500):
    tag = f'{pyramid}-dl{ndl}-q{nq}-dn{nd}'
    cfg_path, init_path = ARCH / f'rtdetr-{tag}.yaml', ARCH / f'rtdetr-{tag}-init.pt'
    if not init_path.exists():
        neck, outs, warm = PYRAMIDS[pyramid]
        cfg = dict(BASE_YAML, head=BASE_YAML['head'][:12] + neck + [[outs, 1, 'RTDETRDecoder', ['nc', 256, nq, 4, 8, ndl, 1024, 0.0, None, -1, nd]]])
        yaml.safe_dump(cfg, open(cfg_path, 'w'))
        src = RTDETR('rtdetr-l.pt').model
        dst = RTDETRDetectionModel(str(cfg_path), nc=src.yaml['nc'], verbose=False)
        idx, lv = {i: i for i in range(22)} | warm | {9 + len(cfg['head']): 28}, [0] * (len(outs) - 3) + [0, 1, 2]
        ss, out = src.state_dict(), {}
        for k, v in dst.state_dict().items():
            p = k.split('.')
            if int(p[1]) not in idx:
                continue
            p[1] = str(idx[int(p[1])])
            if p[2] == 'input_proj':
                p[3] = str(lv[int(p[3])])
            w = ss.get('.'.join(p))
            if w is not None and w.shape != v.shape and p[-2] in ('sampling_offsets', 'attention_weights'):
                w = w.reshape(8, 3, -1, *w.shape[1:])[:, lv].reshape(v.shape)
            if w is not None and w.shape == v.shape:
                out[k] = w
        dst.load_state_dict(out, strict=False)
        dst.names = src.names
        torch.save({'model': dst, 'train_args': {}}, init_path)
        print(f'{tag}: {len(out)}/{len(dst.state_dict())} tensors initialised from rtdetr-l.pt')
    return str(cfg_path), str(init_path)

def train_arch(name, pyramid='p3p5', ndl=6, data=f'fit{TRAIN_STEP}', epochs=EPOCHS, **kw):
    weights = RUNS / name / 'weights' / 'last.pt'
    done = weights.exists() and torch.load(weights, map_location='cpu', weights_only=False)['epoch'] == -1
    if not done:
        check_time()
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        per_gpu = 4
        while True:
            gpus = dict(device=list(range(GPUS)), batch=per_gpu * GPUS) if GPUS > 1 else dict(batch=-1)
            gc.collect(); torch.cuda.empty_cache()
            try:
                if weights.exists():
                    RTDETR(weights).train(resume=True, **gpus)
                else:
                    cfg, init = build_arch(pyramid, ndl)
                    RTDETR(cfg).load(init).train(**dict(data=str(DS / f'{data}.yaml'), project=PROJECT, name=name, save_dir=str((RUNS / name).resolve()), imgsz=IMGSZ,
                                                        epochs=epochs, close_mosaic=3, seed=SEED, workers=4) | gpus | kw)
                break
            except subprocess.CalledProcessError:
                if GPUS < 2 or per_gpu == 1:
                    raise
                per_gpu //= 2
                print(f'{name}: DDP training failed, retrying with {per_gpu} images per GPU')
    return RTDETR(weights)

def arch_stats(name):
    m = RTDETR(RUNS / name / 'weights' / 'last.pt').model
    return {'params, M': round(sum(p.numel() for p in m.parameters()) / 1e6, 2), 'GFLOPs': round(get_flops(m, IMGSZ), 1)}

def size_bins(g):
    return pd.cut(g.side_in, [0, 16, 32, 64, np.inf], labels=['<16 px', '16-32 px', '32-64 px', '>64 px'])

for p in PYRAMIDS:
    build_arch(p)

#### A1. How many decoder layers the baseline needs

In [ ]:
def depth_eval(k):
    m = train_ultra(best_rtdetr, **rtdetr_cfg)
    m.model.model[-1].decoder.eval_idx = k - 1
    dets, ms = predict_ultra(m, VAL)
    return evaluate(dets) | {'ms/img': round(ms, 1)}

depth = pd.DataFrame({k: cached(ARCH / f'depth_{k}.json', lambda: depth_eval(k)) for k in range(1, 7)}).T.rename_axis('decoder layers')
NDL = int(min(max(depth.index[depth.mAP >= depth.loc[6, 'mAP'] - 0.005].min(), 3), 5))
fig, ax = plt.subplots(figsize=(8, 4))
for col, c in zip(['mAP', 'AP50', 'AP_small'], COLORS):
    ax.plot(depth.index, depth[col], 'o-', color=c, label=col)
ax2 = ax.twinx()
ax2.plot(depth.index, depth['ms/img'], 's--', color='gray', label='ms/img')
ax.set(xlabel='decoder layers used at inference', ylabel='validation AP', title=f'{best_rtdetr}: stopping the decoder early')
ax2.set_ylabel('ms/img')
fig.legend(loc='lower right', bbox_to_anchor=(0.88, 0.15))
plt.tight_layout()
wb_log('bonus_arch', decoder_depth=fig, decoder_depth_table=depth)
plt.show()
print('decoder layers to train in E3:', NDL)
depth[METRICS + ['ms/img']]

#### A2. Which feature levels the queries come from

In [ ]:
@torch.no_grad()
def query_levels(m, fr, thr=0.3):
    m.predict(fr.path.iloc[0], verbose=False)
    net = m.predictor.model.model
    head, store, sel, tok, hits = net.model[-1], {}, [], [], []
    hooks = [net.model[0].register_forward_hook(lambda mod, a, o: store.update(inp=a[0].shape[-2:])),
             head.enc_score_head.register_forward_hook(lambda mod, a, o: store.update(enc=o[0].max(-1).values)),
             head.register_forward_hook(lambda mod, a, o: store.update(shapes=[x.shape[-2:] for x in a[0]], boxes=o[1][0][0, 0], logits=o[1][1][0, 0]))]
    for f in fr.itertuples():
        m.predict(f.path, conf=CONF, iou=NMS_IOU, max_det=MAX_DET, verbose=False)
        H, W = store['inp']
        sizes = torch.tensor([h * w for h, w in store['shapes']])
        names = [f'P{int(round(np.log2(W / w)))}' for _, w in store['shapes']]
        ind = store['enc'].topk(min(head.num_queries, len(store['enc']))).indices.cpu()
        lvl = torch.bucketize(ind, sizes.cumsum(0), right=True).numpy()
        sel.append(np.bincount(lvl, minlength=len(names)))
        tok.append(sizes.numpy() / sizes.sum().item())
        score, label = store['logits'].sigmoid().max(-1)
        keep = (score >= thr).cpu().numpy()
        g = objs[objs.image_id == f.image_id]
        if len(g) == 0 or not keep.any():
            continue
        b = store['boxes'].cpu().numpy()[keep] * [f.width, f.height, f.width, f.height]
        iou = box_iou(torch.tensor(np.c_[b[:, :2] - b[:, 2:] / 2, b[:, :2] + b[:, 2:] / 2]), torch.tensor(xyxy(g[['x', 'y', 'w', 'h']].values))).numpy()
        iou *= label.cpu().numpy()[keep][:, None] == g.cls.values[None]
        ok = iou.max(1) >= .5
        hits.append(pd.DataFrame({'level': np.array(names)[lvl[keep][ok]], 'side_in': g.side_in.values[iou.argmax(1)[ok]]}))
    for h in hooks:
        h.remove()
    h = pd.concat(hits)
    share = pd.DataFrame({'feature tokens': np.mean(tok, 0), 'selected queries': np.sum(sel, 0) / np.sum(sel), 'correct detections': h.level.value_counts(normalize=True).reindex(names).fillna(0).values}, index=names)
    return share.round(3), (pd.crosstab(size_bins(h), h.level, normalize='index') * 100).round(1)

LEVEL_FRAMES = VAL.sample(60, random_state=SEED)
base_share, base_by_size = query_levels(train_ultra(best_rtdetr, **rtdetr_cfg), LEVEL_FRAMES)
fig, ax = plt.subplots(figsize=(7, 3.5))
base_share.plot.bar(ax=ax, rot=0, color=COLORS[:3], title=f'{best_rtdetr}: where queries and correct detections come from')
ax.set(ylabel='share')
plt.tight_layout()
wb_log('bonus_arch', query_levels=fig, query_level_share=base_share, query_level_by_size=base_by_size)
plt.show()
display(base_share)
base_by_size

#### E1-E4. Training the architecture changes under the shared protocol

In [ ]:
ARCH_RUNS = {'arch_seed1': dict(pyramid='p3p5', ndl=6, seed=1),
             'arch_p2p4': dict(pyramid='p2p4', ndl=6),
             'arch_p2p5': dict(pyramid='p2p5', ndl=6),
             f'arch_dl{NDL}': dict(pyramid='p3p5', ndl=NDL)}
for name, c in ARCH_RUNS.items():
    run_eval(name, train_arch(name, **c))
NOISE = round(abs(results['arch_seed1']['mAP'] - results[best_rtdetr]['mAP']), 4)
arch_table = pd.concat([table([best_rtdetr, *ARCH_RUNS]), pd.DataFrame({n: arch_stats(n) for n in [best_rtdetr, *ARCH_RUNS]}).T], axis=1)
wb_log('bonus_arch', experiments=arch_table)
print(f'seed noise |mAP(seed 42) - mAP(seed 1)| = {NOISE}')
arch_table

#### What changes between the baseline and the new pyramids

In [ ]:
ARCH_NAMES = [best_rtdetr, 'arch_p2p4', 'arch_p2p5', f'arch_dl{NDL}']
fig, ax = plt.subplots(figsize=(8, 4))
for n, c in zip([best_rtdetr, *ARCH_RUNS], COLORS):
    ax.plot(*curve(n), 'o-', color=c, ms=5, lw=2, label=n)
ax.set(xlabel='epoch', ylabel='validation mAP', title='validation mAP during training')
ax.legend()
plt.tight_layout()
wb_log('bonus_arch', arch_curves=fig)
plt.show()

arch_density = pd.DataFrame(cached(ARCH / 'density.json', lambda: {n: {d: evaluate(load_dets(n), n_val.index[density == d])['mAP'] for d in density.cat.categories} for n in ARCH_NAMES})).T.add_prefix('mAP ')
arch_recall = pd.DataFrame({n: label_errors(n)[1].pipe(lambda g: g.groupby(size_bins(g), observed=True).found.mean()) for n in ARCH_NAMES}).T.add_prefix('recall ')
per_class_arch = pd.DataFrame({n: {c: results[n][c] for c in CLASSES} for n in ARCH_NAMES}).T
wb_log('bonus_arch', arch_density=arch_density, arch_recall=arch_recall, arch_per_class=per_class_arch)
display((pd.concat([table(ARCH_NAMES)[['mAP', 'AP_small', 'AP_medium', 'AP_large']], arch_recall * 100, arch_density], axis=1)).round(3))
display(per_class_arch.round(3))
compare_errors(ARCH_NAMES)

In [ ]:
level_share, level_by_size = {best_rtdetr: base_share}, {best_rtdetr: base_by_size}
for n in ['arch_p2p4', 'arch_p2p5']:
    level_share[n], level_by_size[n] = query_levels(RTDETR(RUNS / n / 'weights' / 'last.pt'), LEVEL_FRAMES)
level_share, level_by_size = pd.concat(level_share), pd.concat(level_by_size).fillna(0)
wb_log('bonus_arch', level_share_all=level_share, level_by_size_all=level_by_size)
display(level_share)
level_by_size

#### E5. Final architecture and inference scale

In [ ]:
PYR_RUN = {'p3p5': best_rtdetr, 'p2p4': 'arch_p2p4', 'p2p5': 'arch_p2p5'}
base_map = results[best_rtdetr]['mAP']
pyr = max(PYR_RUN, key=lambda p: results[PYR_RUN[p]]['mAP'])
pyr = pyr if results[PYR_RUN[pyr]]['mAP'] > base_map + NOISE else 'p3p5'
shallow = results[f'arch_dl{NDL}']['mAP'] >= base_map - NOISE and results[f'arch_dl{NDL}']['ms/img'] < results[best_rtdetr]['ms/img']
FINAL_ARCH = dict(pyramid=pyr, ndl=NDL if shallow else 6)
ARCH_RUN = {('p3p5', 6): best_rtdetr, ('p2p4', 6): 'arch_p2p4', ('p2p5', 6): 'arch_p2p5', ('p3p5', NDL): f'arch_dl{NDL}'}.get(tuple(FINAL_ARCH.values()), 'arch_combo')
if ARCH_RUN == 'arch_combo':
    run_eval(ARCH_RUN, train_arch(ARCH_RUN, **FINAL_ARCH))
print('pyramid kept:', pyr, '| shallow decoder kept:', shallow, '| final architecture:', FINAL_ARCH, '->', ARCH_RUN)

agrid = {}
for n in dict.fromkeys([best_rtdetr, ARCH_RUN]):
    for s, scale in SCALES.items():
        agrid[n, s] = cached(RUNS / 'pipeline' / f'{n} {s}.json', lambda: timed_eval(lambda: predict_scaled(RTDETR(RUNS / n / 'weights' / 'last.pt'), VAL, scale)))
ARCH_SCALE = max([k for k in agrid if k[0] == ARCH_RUN], key=lambda k: agrid[k]['mAP'])[1]
reference = grid[best_pipe, FINAL_STRATEGY]['mAP']
USE_ARCH = ARCH_RUN != best_rtdetr and agrid[ARCH_RUN, ARCH_SCALE]['mAP'] > reference + NOISE
print(f'best scale for {ARCH_RUN}: {ARCH_SCALE} ({agrid[ARCH_RUN, ARCH_SCALE]["mAP"]:.4f}) | current final model: {best_pipe}, {FINAL_STRATEGY} ({reference:.4f}) | replace it: {USE_ARCH}')
arch_scales = pd.DataFrame(agrid).T[METRICS + ['ms/img']]
wb_log('bonus_arch', arch_scales=arch_scales)
arch_scales

#### Failure cases of the final architecture

In [ ]:
dets, gts = error_plots(ARCH_RUN)
show_cases(dets, gts)

#### Submission with the final architecture

In [ ]:
if USE_ARCH:
    arch_setup = FINAL_ARCH | {'data': f'all{TRAIN_STEP}'}
    if (ARCH / 'final_arch.json').exists() and json.load(open(ARCH / 'final_arch.json')) != arch_setup:
        shutil.rmtree(RUNS / 'arch_final', ignore_errors=True)
    json.dump(arch_setup, open(ARCH / 'final_arch.json', 'w'))
    arch_final = train_arch('arch_final', **FINAL_ARCH, data=f'all{TRAIN_STEP}', val=False)
    arch_test = cached(RUNS / 'arch_final' / f'test_preds {ARCH_SCALE}.json', lambda: predict_scaled(arch_final, test_fr, SCALES[ARCH_SCALE]))
    submission(arch_test, test.image_id).to_csv('submission_arch.csv', index=False)
    print('written: submission_arch.csv | architecture:', FINAL_ARCH, '| inference:', ARCH_SCALE)
    display(final_candidates('arch_pipeline', ARCH_RUN, arch_final, ARCH_SCALE, None, arch_test))
else:
    print(f'no architecture change beats {best_pipe} ({FINAL_STRATEGY}) by more than the seed noise; the final model stays the same')

### What the architecture experiments show